# MedAssist — Fine-tune QLoRA bước chọn trích xuất triệu chứng (Kaggle GPU 16 GB)

- **Mục tiêu:** Fine-tune riêng bước **chọn** của trích xuất `phrases` (`extraction-phrases-v2` + `extraction-phrases-compact-v3`) theo `fine-tune-trich-xuat.md` bản 3.
- **Model gốc:** `aisingapore/Gemma-SEA-LION-v3-9B-IT` (QLoRA 4-bit với Unsloth).
- **Đầu vào cần đính kèm từ Kaggle Dataset:** `train.jsonl`, `val.jsonl`, `dev.jsonl`, `config.json` (tạo từ `training/extraction/generate_dataset.py` sau khi Tuấn đóng băng `evaluation/testset_extraction_ft/`).
- **Đầu ra:** GGUF `Q4_K_M` (`medassist-extraction-q4_k_m.gguf`) + SHA-256 để nạp vào Ollama qua `training/extraction/Modelfile` (`OLLAMA_EXTRACTION_MODEL`).

In [ ]:
!pip install -q --no-deps bitsandbytes accelerate xformers==0.0.29.post3 peft trl triton cut_cross_entropy unsloth_zoo
!pip install -q sentencepiece protobuf "datasets>=3.4.1" huggingface_hub hf_transfer
!pip install -q --no-deps unsloth
!pip install -q --no-deps llama-cpp-python || true

In [ ]:
import hashlib
import json
from pathlib import Path
import torch

# Sửa DATA_DIR theo tên Kaggle Dataset bạn đã upload
CANDIDATE_DIRS = [
    Path("/kaggle/input/medassist-extraction-ft"),
    Path("/kaggle/working/data"),
    Path("./data"),
]
DATA_DIR = next((d for d in CANDIDATE_DIRS if (d / "train.jsonl").exists()), CANDIDATE_DIRS[0])
CFG_PATH = DATA_DIR / "config.json" if (DATA_DIR / "config.json").exists() else Path("config.json")
OUT_DIR = Path("/kaggle/working/medassist_extraction_out")
OUT_DIR.mkdir(parents=True, exist_ok=True)

cfg = json.loads(CFG_PATH.read_text(encoding="utf-8")) if CFG_PATH.exists() else {
    "seed": 20261122,
    "qlora": {
        "base_model": "aisingapore/Gemma-SEA-LION-v3-9B-IT",
        "load_in_4bit": True,
        "max_seq_length": 4096,
        "r": 16,
        "lora_alpha": 16,
        "lora_dropout": 0.0,
        "target_modules": ["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
        "learning_rate": 2e-4,
        "num_train_epochs": 2,
        "per_device_train_batch_size": 2,
        "gradient_accumulation_steps": 8,
        "warmup_ratio": 0.05,
        "lr_scheduler_type": "cosine",
        "optim": "adamw_8bit",
        "quantization_method": "q4_k_m",
    },
}
qcfg = cfg["qlora"]
print("DATA_DIR:", DATA_DIR, "| GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU")
for name in ("train.jsonl", "val.jsonl", "dev.jsonl"):
    p = DATA_DIR / name
    print(f"  {name}: exists={p.exists()}, lines={sum(1 for l in p.open(encoding='utf-8') if l.strip()) if p.exists() else 0}")

In [ ]:
from unsloth import FastLanguageModel

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=qcfg["base_model"],
    max_seq_length=qcfg["max_seq_length"],
    dtype=None,
    load_in_4bit=qcfg["load_in_4bit"],
)
model = FastLanguageModel.get_peft_model(
    model,
    r=qcfg["r"],
    target_modules=qcfg["target_modules"],
    lora_alpha=qcfg["lora_alpha"],
    lora_dropout=qcfg["lora_dropout"],
    bias="none",
    use_gradient_checkpointing="unsloth",
    random_state=cfg["seed"],
)

In [ ]:
from datasets import Dataset

def format_gemma_chat(example):
    """Khớp chính xác Modelfile Ollama của Gemma-SEA-LION-v3-9B-IT-GGUF:Q4_K_M."""
    msgs = example["messages"]
    user_text = msgs[0]["content"]
    assistant_text = msgs[1]["content"]
    return {
        "text": (
            f"<start_of_turn>user\n{user_text}<end_of_turn>\n"
            f"<start_of_turn>model\n{assistant_text}<end_of_turn>\n"
        )
    }

def load_jsonl_dataset(path: Path) -> Dataset:
    rows = [json.loads(line) for line in path.read_text(encoding="utf-8").splitlines() if line.strip()]
    ds = Dataset.from_list([{"id": r["id"], "messages": r["messages"]} for r in rows])
    return ds.map(format_gemma_chat, remove_columns=["messages"])

train_ds = load_jsonl_dataset(DATA_DIR / "train.jsonl")
val_ds = load_jsonl_dataset(DATA_DIR / "val.jsonl")
print("train:", len(train_ds), "| val:", len(val_ds))
print("sample prefix:", repr(train_ds[0]["text"][:120]))

In [ ]:
from transformers import TrainingArguments, DataCollatorForSeq2Seq
from trl import SFTTrainer
from unsloth import is_bfloat16_supported
from unsloth.chat_templates import train_on_responses_only

trainer = SFTTrainer(
    model=model,
    tokenizer=tokenizer,
    train_dataset=train_ds,
    eval_dataset=val_ds,
    dataset_text_field="text",
    max_seq_length=qcfg["max_seq_length"],
    data_collator=DataCollatorForSeq2Seq(tokenizer=tokenizer),
    dataset_num_proc=2,
    packing=False,
    args=TrainingArguments(
        output_dir=str(OUT_DIR / "checkpoints"),
        per_device_train_batch_size=qcfg["per_device_train_batch_size"],
        gradient_accumulation_steps=qcfg["gradient_accumulation_steps"],
        warmup_ratio=qcfg["warmup_ratio"],
        num_train_epochs=qcfg["num_train_epochs"],
        learning_rate=qcfg["learning_rate"],
        fp16=not is_bfloat16_supported(),
        bf16=is_bfloat16_supported(),
        logging_steps=10,
        eval_strategy="steps",
        eval_steps=50,
        save_strategy="steps",
        save_steps=50,
        save_total_limit=2,
        load_best_model_at_end=True,
        metric_for_best_model="eval_loss",
        optim=qcfg["optim"],
        lr_scheduler_type=qcfg["lr_scheduler_type"],
        seed=cfg["seed"],
        report_to="none",
    ),
)

trainer = train_on_responses_only(
    trainer,
    instruction_part="<start_of_turn>user\n",
    response_part="<start_of_turn>model\n",
)

# Kiểm nhanh rằng chỉ phần assistant được tính loss
masked_len = sum(1 for x in trainer.train_dataset[0]["labels"] if x == -100)
total_len = len(trainer.train_dataset[0]["labels"])
print(f"Token masked (-100): {masked_len}/{total_len} | Response tokens: {total_len - masked_len}")

train_result = trainer.train()
eval_result = trainer.evaluate()
(OUT_DIR / "train_metrics.json").write_text(
    json.dumps({"train": train_result.metrics, "eval": eval_result}, ensure_ascii=False, indent=2),
    encoding="utf-8",
)
print("Eval metrics:", eval_result)

In [ ]:
# Kiểm nhanh độ hợp lệ JSON và khớp chính xác trên mẫu dev.jsonl trước khi xuất GGUF
FastLanguageModel.for_inference(model)
dev_rows = [json.loads(line) for line in (DATA_DIR / "dev.jsonl").read_text(encoding="utf-8").splitlines() if line.strip()][:40]

json_ok = 0
exact_ok = 0
for r in dev_rows:
    user_prompt = r["messages"][0]["content"]
    gold = json.loads(r["messages"][1]["content"])
    prompt_str = f"<start_of_turn>user\n{user_prompt}<end_of_turn>\n<start_of_turn>model\n"
    inputs = tokenizer([prompt_str], return_tensors="pt").to(model.device)
    with torch.no_grad():
        gen = model.generate(**inputs, max_new_tokens=768, do_sample=False, temperature=None, top_p=None)
    ans = tokenizer.decode(gen[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True).split("<end_of_turn>")[0].strip()
    try:
        pred = json.loads(ans)
        json_ok += 1
        if pred == gold:
            exact_ok += 1
    except Exception:
        pass

print(f"Dev sample (n={len(dev_rows)}): valid_json={json_ok}/{len(dev_rows)} | exact_json={exact_ok}/{len(dev_rows)}")

In [ ]:
lora_dir = OUT_DIR / "lora_adapter"
gguf_dir = OUT_DIR / "gguf"
model.save_pretrained(str(lora_dir))
tokenizer.save_pretrained(str(lora_dir))

# Xuất GGUF Q4_K_M; đổi tên về medassist-extraction-q4_k_m.gguf để khớp training/extraction/Modelfile
model.save_pretrained_gguf(
    str(gguf_dir),
    tokenizer,
    quantization_method=qcfg["quantization_method"],
)

gguf_files = sorted(gguf_dir.glob("*.gguf")) + sorted(Path(".").glob("*Q4_K_M*.gguf"))
for gf in gguf_files:
    h = hashlib.sha256()
    with gf.open("rb") as f:
        for chunk in iter(lambda: f.read(1024 * 1024), b""):
            h.update(chunk)
    print(f"GGUF: {gf} | size={gf.stat().st_size / (1024**3):.2f} GB | sha256={h.hexdigest()}")